# Первое домашнее задание. Толстых Александра НПМбд-02-24

In [13]:
# импорт необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import mean_squared_error

## Основной код

In [ ]:
# ЧАСТЬ А

# Чтение
img = Image.open('pic1.jpg')
img_gray = img.convert('L')  # Градации серого
A = np.array(img_gray)
h, w = A.shape
original_shape = (h, w)

original_flat = A.flatten()

print("Исходный размер:", original_shape)


# ЧАСТЬ Б
# SVD
U, s, Vt = np.linalg.svd(A, full_matrices=False)
list_of_ranks =  [1, 5, 10, 30, 100, min(h, w)]
approximations = []
for r in list_of_ranks:
    A_approx = (U[:, :r] * s[:r]) @ Vt[:r, :]
    A_approx = np.clip(A_approx, 0, 255).astype(np.uint8)
    approximations.append(A_approx)


# ЧАСТЬ B
# Визуализация картинок на подграфиках
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# ЧАСТЬ Г
for idx, (r, img_comp) in enumerate(zip(list_of_ranks, approximations)):
    orig_size = h * w
    comp_size = r * (h + w + 1) * 4
    comp_ratio = orig_size / comp_size

    axes[idx].imshow(img_comp, cmap='gray')
    axes[idx].set_title(f"Ранг = {r}\nСжатие в {comp_ratio:.2f} раз")
    axes[idx].axis('off')


plt.axis('off')
plt.show()

# Ориг картинка
plt.figure(figsize=(6, 6))
plt.imshow(A, cmap='gray')
plt.title("Оригинальное изображение (без сжатия)")
plt.axis('off')
plt.show()

Исходный размер: (1329, 886)


## Автопроверка

In [ ]:
# ---------- БЛОК АВТОПРОВЕРКИ (НЕ РЕДАКТИРОВАТЬ) ----------
# Предполагается, что у вас есть переменные:
# original_shape = (h, w, c) или (h, w)
# list_of_ranks = [1, 5, 10, 30, 100, ...]
# approximations = список восстановленных массивов (numpy) для каждого ранга

# 1. Проверка размерностей
for i, r in enumerate(list_of_ranks):
    assert approximations[i].shape == original_shape, f"Ошибка: размерность для ранга {r} не совпадает с исходной"

# 2. Проверка, что значения не выходят за пределы 0-255 (для RGB/серого)
for i, r in enumerate(list_of_ranks):
    arr = approximations[i]
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255) # Если float, то проверяем диапазон
    assert arr.max() <= 255.1, f"Значения превышают 255 для ранга {r}"
    assert arr.min() >= -0.1, f"Значения меньше 0 для ранга {r}"

# 3. Проверка, что с ростом ранга ошибка уменьшается (метрика MSE)
from sklearn.metrics import mean_squared_error
mse_list = []
for i in range(len(approximations)):
    # Если цветная, считаем MSE по всем каналам
    mse = mean_squared_error(original_flat, approximations[i].flatten())
    mse_list.append(mse)

# Проверяем монотонность (MSE должен падать, так как мы добавляем сингулярные числа)
for i in range(1, len(mse_list)):
    assert mse_list[i] <= mse_list[i-1] + 1e-6, f"MSE не уменьшается между рангами {list_of_ranks[i-1]} и {list_of_ranks[i]}"

print("✅ Все автоматические проверки пройдены. Задание выполнено корректно!")

## Результаты

***Результаты части А:***

Загружено изображение pic1.jpg.

Исходное изображение имеет размер 1329×886 пикселей и было преобразовано в режим градаций серого (1 канал).

Изображение соответствует требованиям ТЗ, так как его размер превышает 300×300 пикселей.

***Результаты части Б:***

Выполнено SVD-разложение изображения с параметром:
full_matrices=False

Восстановлены изображения для следующих рангов: 1, 5, 10, 30, 100, 886

Последний ранг равен: min(h, w) = min(1329, 886) = 960

Все восстановленные изображения имеют размер 1329×886 пикселей, совпадающий с размером исходного изображения.

Значения пикселей ограничены диапазоном от 0 до 255 и преобразованы к типу uint8.

***Результаты части В:***

Построена визуализация в виде сетки 2×3 с восстановленными изображениями различных рангов.

Для каждого изображения указаны его ранг и коэффициент сжатия.

Получены следующие коэффициенты сжатия:

r = 1: 132.84 раз

r = 5: 26.57 раз

r = 10: 13.28 раз

r = 30: 4.43 раз

r = 100: 1.33 раз

r = 886: 0.15 раз

Оригинальное изображение выведено отдельно.

***Результаты части Г:***

Анализ результатов показывает, что с увеличением ранга качество восстановления постепенно улучшается, а значение ошибки MSE уменьшается.

При низких рангах достигается сильное сжатие, однако изображение заметно теряет детали.

При увеличении ранга изображение становится всё более похожим на оригинал.


## Выводы

1. SVD-разложение позволяет восстанавливать изображение с различным уровнем качества путём выбора количества используемых сингулярных чисел.
2. При низких рангах достигается значительное сжатие изображения, но происходит заметная потеря деталей.
3. При r = 30 изображение уже достаточно хорошо передаёт основные объекты и структуру исходной фотографии при коэффициенте сжатия 4.43 раза.
4. При r = 100 качество изображения визуально близко к оригиналу, при этом коэффициент сжатия составляет 1.33 раза.
5. При максимальном ранге r = 886 достигается практически полное восстановление изображения.
6. С увеличением ранга значение MSE монотонно уменьшается, что подтверждает улучшение качества восстановления.

Таким образом, SVD позволяет эффективно управлять компромиссом между степенью сжатия и качеством изображения.